In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import plotly.graph_objects as go
import numpy as np
from plotly.io import write_html

In [12]:
log2_path = "../results/deseq2_rsem_sortmerna/hydra_all_deseq2_lg2_t_to_g.tsv"
pval_path = "../results/deseq2_rsem_sortmerna/hydra_all_deseq2_pvalues_t_to_g.tsv"
annotation_path = "../data/annotation_KO_GO.csv"
annotation_path_single_cell_atlas = "../data/HVAEP1_annotation.csv"
sample_description_path = "../data/rsem_counts_after_sortmerna/sample_descriptions.csv"
unique_celltypes_path = "../results/processedData/normalized_mean_sortmerna/unique_celltypes.table"

In [13]:
'''read_unique_celltype_table
    
    This function reads the unique celltype table.
    
    :param unique_celltype_path
        :type str
    
    :returns unique_celltypes
        :type pd.DataFrame

'''
def read_unique_celltype_table(unique_celltype_path:str)->pd.DataFrame:
    try:
        print("[*] Reading unique celltype dataframe ...")
        unique_celltypes = pd.read_table(unique_celltypes_path ,sep=";")
        # transforming ID column 
        print("[*] Transforming ID column ...")
        unique_celltypes = unique_celltypes[unique_celltypes["ID"].isna() == False]
        unique_celltypes["ID"] = unique_celltypes["ID"].apply(lambda x: x.split("-")[1])
        print("[+] DONE")
        return unique_celltypes
    except Exception as e:
        raise Exception("[-] ERROR reading unique celltype table with exception: {}".format(e))

In [14]:
'''build_celltype_dictionary

    This function parses through the columns of the unqiue_celltypes dataframe
    and filters the gene IDs for each celltype. The IDs are saved in a dictionary with
    celltypes as keys and ids as values.
    
    :param unique_celltypes
        :type pd.DataFrame
    
    :returns celltype_dict
        :type dict[str] = list

'''
def build_celltype_dictionary(unique_celltypes:pd.DataFrame)->dict:
    try:
        print("[*] Creating Celltype Dictionary")
        celltype_dict = {}
        for celltype in unique_celltypes.columns:
            if celltype != "ID":
                current_celltype = unique_celltypes[unique_celltypes[celltype] == 1.0]
                celltype_dict[celltype] = current_celltype.ID.to_list()
                print("\t[*] Working on: {} with {} number of genes ...".format(celltype, len(current_celltype)))
        print("[+] DONE")
        return celltype_dict
    except Exception as e:
        raise Exception("[-] ERROR creating celltype dictionary with exception: {}".format(e))

In [15]:
'''read_differential_expression_tables
    
    This function reads the comprehensive log2FoldChange and pvalue dataframes, in which all 
    transcriptomic data resides. It then transforms the ID column to a gene_id column for further
    merging steps. This will ensure same gene identifiers as strings across different dataframes.
    
    :param log2FoldChange_path
        :type str
    :param pValue_path
        :type str
        
    :returns diff_exp_table, p_value_table
        :type tupe(pd.DataFrame, pd.DataFrame)

'''
def read_differential_expression_tables(log2FoldChange_path:str,pValue_path:str)->tuple:
    try:
        
        print("[*] Reading dataframes ...")
        diff_exp_table = pd.read_csv(log2FoldChange_path, sep="\t")
        p_value_table = pd.read_csv(pValue_path, sep="\t")
        # renaming ID column for merging
        print("[*] Transforming ID column ...")
        p_value_table.rename(columns={"ID":"gene_id"}, inplace=True)
        diff_exp_table.rename(columns={"ID":"gene_id"}, inplace=True)
        # equalize gene_ids for merging
        diff_exp_table["gene_id"] = diff_exp_table["gene_id"].apply(lambda x: x.split("-")[1])
        p_value_table["gene_id"] = p_value_table["gene_id"].apply(lambda x: x.split("-")[1])
        
        print("[*] Transforming NA values to none-differentially expressed genes ...")
        # handling missing data as non differentially expressed genes
        diff_exp_table = diff_exp_table.fillna(0.0)
        p_value_table = p_value_table.fillna(1.0)
        
        print("[+] DONE")
        return diff_exp_table, p_value_table
    except Exception as e:
        raise Exception("[-] ERROR reading transcriptome tables with exception: {}".format(e)) 

In [16]:
'''read_annotation_file_single_cell_atlas

    This function reads the HVAEP1_annotation.csv file from the brown_hydra_genomes GitHub repository.
    
    :param annotation_path
        :type str
    
    :returns annotation_dataframe
        :type pd.DataFrame

'''
def read_annotation_file_single_cell_atlas(annotation_path:str)->pd.DataFrame:
    try:
        print("[*] Reding brown hydra genome annotation file ...")
        annotation_dataframe = pd.read_csv(annotation_path)
        annotation_dataframe.rename(columns={"H_vulgarisAEP":"gene_id"}, inplace=True)
        annotation_dataframe["gene_id"] = annotation_dataframe["gene_id"].apply(lambda x: x.split("_")[-1].replace("T","G").split(".")[0])
        annotation_dataframe = annotation_dataframe.drop_duplicates(subset="gene_id", keep="first")
        print("[+] DONE")
        return annotation_dataframe
    except Exception as e:
        raise Exception("[-] ERROR reading annotation file with exception: {}".format(e))

In [17]:
'''read_annotation_table

    This function reads the annotation table, transforms the ID column and returns the dataframe.
    
    :param annot_path
        :type str
    
    :returns annotation_table
        :type pd.DataFrame
'''
def read_annotation_table(annot_path:str)->pd.DataFrame:
    try:
        print("[*] Reading annotation table ...")
        annotation_table = pd.read_csv(annot_path)
        print("[*] Transforming ID column ...")
        # just work with valid IDs
        annotation_table = annotation_table[annotation_table["ID"].isna() == False]
        # renaming and euqalizing ID column
        annotation_table["gene_id"] = annotation_table["ID"].apply(lambda x: x.split(".")[1].replace("T","G"))
        print("[+] DONE")
        return annotation_table
    except Exception as e:
        raise Exception("[-] ERROR reading annotation table with exception: {}".format(e))

In [18]:
'''read_sample_description

    This function reads the sample description dataframe.
    
    :param sample_description_path
        :type str
        
    :returns sample_description
        :type pd.DataFrame

'''
def read_sample_description(sample_description_path:str)->pd.DataFrame:
    try:
        print("[*] Reading sample description dataframe ...")
        sample_description = pd.read_csv(sample_description_path)
        print("[+] DONE")
        return sample_description
    except Exception as e:
        raise Exception("[-] ERROR reading sample description table with exception: {}".format(e))

In [19]:
'''transform_transcript_ids_to_gene_ids

    This script transforms transcript identifier to gene identifier by replacing the T with a G.
    
    :param transcript_ids
        :type list
    
    :returns gene_ids
        :type list
'''
def transform_transcript_ids_to_gene_ids(transcript_ids:list)->list:
    try:
        gene_ids = []
        print("[*] Working with: {} transcripts ...".format(len(transcript_ids)))
        print("[*] Starting parsing and renaming ...")
        for transcript in transcript_ids:
            gene = transcript.split("_")[1].replace("T","G")
            gene_ids.append(gene)
        print("[+] DONE") 
        return gene_ids
    except Exception as e:
        raise Exception("[-] ERROR during parsing of transcript IDs with exception: {}".format(e))

In [20]:
unique_celltypes = read_unique_celltype_table(unique_celltypes_path)
celltype_dict = build_celltype_dictionary(unique_celltypes)

[*] Reading unique celltype dataframe ...
[*] Transforming ID column ...
[+] DONE
[*] Creating Celltype Dictionary
	[*] Working on: Ec_Head with 29 number of genes ...
	[*] Working on: En_Head with 11 number of genes ...
	[*] Working on: Ec_BodyCol.SC with 3 number of genes ...
	[*] Working on: I_ISC with 1 number of genes ...
	[*] Working on: Ec_Peduncle with 23 number of genes ...
	[*] Working on: Ec_Tentacle with 31 number of genes ...
	[*] Working on: I_FemGC with 41 number of genes ...
	[*] Working on: Ec_BasalDisk with 83 number of genes ...
	[*] Working on: I_Neuro with 3 number of genes ...
	[*] Working on: I_EarlyNem with 2 number of genes ...
	[*] Working on: I_MaleGC with 30 number of genes ...
	[*] Working on: I_En2N with 12 number of genes ...
	[*] Working on: En_Foot with 17 number of genes ...
	[*] Working on: En_BodyCol.SC with 10 number of genes ...
	[*] Working on: I_ZymoGl with 61 number of genes ...
	[*] Working on: I_GlProgen with 6 number of genes ...
	[*] Working

In [21]:
wnt_genes_portal = ["HVAEP1_T001980", "HVAEP1_T001985", "HVAEP1_T003934", "HVAEP1_T005251", "HVAEP1_T009462", "HVAEP1_T010730", 
             "HVAEP1_T010776", "HVAEP1_T010792", "HVAEP1_T011290", "HVAEP1_T011553", "HVAEP1_T011641", "HVAEP1_T013231",
             "HVAEP1_T016743", "HVAEP1_T021814", "HVAEP1_T021819", "HVAEP1_T021820", "HVAEP1_T022460", "HVAEP1_T022678",
             "HVAEP1_T023071", "HVAEP1_T023618", "HVAEP1_T023882", "HVAEP1_T028556"]

frizzled_genes_portal = ["HVAEP1_T003380","HVAEP1_T003966","HVAEP1_T005729", "HVAEP1_T009158", "HVAEP1_T011153", 
                  "HVAEP1_T011318", "HVAEP1_T024230", "HVAEP1_T024233"]

beta_catenin_genes_portal = ["HVAEP1_T001123","HVAEP1_T006072","HVAEP1_T006715","HVAEP1_T009586",
                             "HVAEP1_T010630","HVAEP1_T012073","HVAEP1_T016451","HVAEP1_T020303",
                             "HVAEP1_T022033","HVAEP1_T027708"]

#lectin_genes_portal = ["HVAEP1_T018169","HVAEP1_T020564", "HVAEP1_T011832","HVAEP1_T000027","HVAEP1_T016707",
#"HVAEP1_T024231","HVAEP1_T024232","HVAEP1_T004003","HVAEP1_T018345","HVAEP1_T013788",
#"HVAEP1_T027285","HVAEP1_T015928","HVAEP1_T022360","HVAEP1_T010302","HVAEP1_T017444",
#"HVAEP1_T023780","HVAEP1_T018750","HVAEP1_T018751","HVAEP1_T018763", "HVAEP1_T018783"]

lectin_genes_portal = ["HVAEP1_T009292", "HVAEP1_T009293", "HVAEP1_T002089","HVAEP1_T020720", "HVAEP1_T020721", "HVAEP1_T020722","HVAEP1_T026974","HVAEP1_T027433"]

In [22]:
# reading differential expression tables
#diff_exp_table = pd.read_csv("../results/deseq2_rsem/hydra_all_counts.tsv", sep="\t")
#diff_exp_table = pd.read_csv("../results/deseq2_rsem/hydra_all_deseq2_lg2_t_to_g.tsv", sep="\t")
#p_value_table = pd.read_csv("../results/deseq2_rsem/hydra_all_deseq2_pvalues_t_to_g.tsv", sep="\t")
diff_exp_table, p_value_table = read_differential_expression_tables(log2_path, pval_path)
# annotation_table = read_annotation_table(annotation_path)
annotation_table = read_annotation_file_single_cell_atlas(annotation_path_single_cell_atlas)
sample_description = read_sample_description(sample_description_path)

[*] Reading dataframes ...
[*] Transforming ID column ...
[*] Transforming NA values to none-differentially expressed genes ...
[+] DONE
[*] Reding brown hydra genome annotation file ...
[+] DONE
[*] Reading sample description dataframe ...
[+] DONE


In [23]:
wnt_genes = transform_transcript_ids_to_gene_ids(wnt_genes_portal)
frizzled_genes = transform_transcript_ids_to_gene_ids(frizzled_genes_portal)
catenin_genes = transform_transcript_ids_to_gene_ids(beta_catenin_genes_portal)
lectin_genes = transform_transcript_ids_to_gene_ids(lectin_genes_portal)

[*] Working with: 22 transcripts ...
[*] Starting parsing and renaming ...
[+] DONE
[*] Working with: 8 transcripts ...
[*] Starting parsing and renaming ...
[+] DONE
[*] Working with: 10 transcripts ...
[*] Starting parsing and renaming ...
[+] DONE
[*] Working with: 8 transcripts ...
[*] Starting parsing and renaming ...
[+] DONE


In [24]:
'''extract_target_gene_expressions_and_annotations

    This function extracts differential expression data (log2FoldChange, pvalues and annotations) for 
    the provided gene identifiers (list).
    
    :param log2fold_dataframe
        :type pd.DataFrame
    :param pval_dataframe
        :type pd.DataFrame
    :param annotation_dataframe
        :type pd.DataFrame
    :param target_genes
        :type list
    
    :returns log2fold_dataframe, pval_dataframe, annotation_dataframe
        :type tuple(pd.DataFrame, pd.DataFrame, pd.DataFrame)
    
'''
def extract_target_gene_expressions_and_annotations(log2fold_dataframe:pd.DataFrame, 
                                                    pval_dataframe:pd.DataFrame, 
                                                    annotation_dataframe:pd.DataFrame, 
                                                    target_genes:list)->tuple:
    try:
        print("[*] Extracting {} target genes ...".format(len(target_genes)))
        log2fold_dataframe = log2fold_dataframe[log2fold_dataframe["gene_id"].isin(target_genes)]
        pval_dataframe = pval_dataframe[pval_dataframe["gene_id"].isin(target_genes)]        
        if len(log2fold_dataframe) == 0:
            raise Exception("[-] ERROR dataframe length is 0 after filtering for target genes ...")
            
        print("[*] Length after extracting target genes: {} ...".format(len(log2fold_dataframe)))
        print("[*] Extracting KO definition line for annotations ...")
        
        annotation_dataframe = annotation_dataframe[
            annotation_dataframe["gene_id"].isin(target_genes)].drop_duplicates(keep="first", subset="gene_id")
        print("[+] DONE")
        return log2fold_dataframe, pval_dataframe, annotation_dataframe
    except Exception as e:
        raise Exception("[-] ERROR with exception: {}".format(e))


In [25]:
wnt_genes_to_exp, wnt_genes_to_pval, annotation_genes_to_wnt = extract_target_gene_expressions_and_annotations(diff_exp_table, 
                                                                                      p_value_table,
                                                                                      annotation_table, 
                                                                                      wnt_genes)
frizzled_genes_to_exp, frizzled_genes_to_pval, annotation_genes_to_frizzled = extract_target_gene_expressions_and_annotations(diff_exp_table, 
                                                                                      p_value_table,
                                                                                      annotation_table, 
                                                                                      frizzled_genes)
catenin_genes_to_exp, catenin_genes_to_pval, annotation_genes_to_catenin = extract_target_gene_expressions_and_annotations(diff_exp_table, 
                                                                                      p_value_table,
                                                                                      annotation_table, 
                                                                                      catenin_genes)
lectin_genes_to_exp, lectin_genes_to_pval, annotaion_genes_to_lectin = extract_target_gene_expressions_and_annotations(diff_exp_table, 
                                                                                      p_value_table,
                                                                                      annotation_table, 
                                                                                      lectin_genes)

[*] Extracting 22 target genes ...
[*] Length after extracting target genes: 21 ...
[*] Extracting KO definition line for annotations ...
[+] DONE
[*] Extracting 8 target genes ...
[*] Length after extracting target genes: 8 ...
[*] Extracting KO definition line for annotations ...
[+] DONE
[*] Extracting 10 target genes ...
[*] Length after extracting target genes: 10 ...
[*] Extracting KO definition line for annotations ...
[+] DONE
[*] Extracting 8 target genes ...
[*] Length after extracting target genes: 8 ...
[*] Extracting KO definition line for annotations ...
[+] DONE


In [26]:
'''plot_seaborn_heatmap

    This function plots a heatmap of the significant log2Fold values.
    
    :param log2fold_dataframe
        :type pd.DataFrame
    :param pval_dataframe
        :type pd.DataFrame
    :param savep
        :type str
    :param title
        :type str
    
    :returns 0
        :type int

'''
def plot_seaborn_heatmap(log2fold_dataframe:pd.DataFrame, 
                         pval_dataframe:pd.DataFrame, 
                         savep:str, title="Significant Log2Fold values across experiments")->int:
    try:
        print("[*] Starting heatmap plotting procedure ...")
        plt.figure(figsize=(25, 12))
        # gene_id in first column
        custom_y_labels = pval_dataframe["gene_id"]
        mask_matrix = pval_dataframe.iloc[:,1:] >= 0.05  # Example mask matrix
        sns.heatmap(log2fold_dataframe.iloc[:,1:], annot=True, cmap="viridis", mask=mask_matrix, yticklabels=custom_y_labels, annot_kws={"size": 15})
        plt.title(title, fontsize=20)
        plt.tight_layout()
        plt.xticks(fontsize=15)
        print("[*] Saving heatmap into: {}".format(savep))
        plt.savefig(savep, dpi=400)
        plt.close()
        print("[+] DONE")
        return 0
    except Exception as e:
        raise Exception("[-] ERROR during creation of heatmap with exception: {}".format(e))

In [27]:
experiments_to_keep = ["gene_id","HydraRecolonization_Conventionalized_vs_GF","HydraRecolonization_Cvbct_vs_GF","HydraRecolonization_Wild_vs_GF", "HydraRecolonization_Cvbct_vs_Wild"]

In [28]:
lectin_genes_to_exp = lectin_genes_to_exp[experiments_to_keep]
lectin_genes_to_pval = lectin_genes_to_pval[experiments_to_keep]
plot_seaborn_heatmap(lectin_genes_to_exp,
                     lectin_genes_to_pval, 
                     "../results/figures/genes_log2Fold_heatmaps/lectins_log2fold.png")

plot_seaborn_heatmap(frizzled_genes_to_exp,
                     frizzled_genes_to_pval, 
                     "../results/figures/genes_log2Fold_heatmaps/frizzled_log2fold.png")

plot_seaborn_heatmap(catenin_genes_to_exp,
                     catenin_genes_to_pval, 
                     "../results/figures/genes_log2Fold_heatmaps/catenin_log2fold.png")

plot_seaborn_heatmap(wnt_genes_to_exp,
                     wnt_genes_to_pval, 
                     "../results/figures/genes_log2Fold_heatmaps/wnt_log2fold.png")

[*] Starting heatmap plotting procedure ...
[*] Saving heatmap into: ../results/figures/genes_log2Fold_heatmaps/lectins_log2fold.png
[+] DONE
[*] Starting heatmap plotting procedure ...
[*] Saving heatmap into: ../results/figures/genes_log2Fold_heatmaps/frizzled_log2fold.png
[+] DONE
[*] Starting heatmap plotting procedure ...
[*] Saving heatmap into: ../results/figures/genes_log2Fold_heatmaps/catenin_log2fold.png
[+] DONE
[*] Starting heatmap plotting procedure ...
[*] Saving heatmap into: ../results/figures/genes_log2Fold_heatmaps/wnt_log2fold.png
[+] DONE


0

In [29]:
'''plot_goHeatmap_html

    This function crates an interactive HTML heatmap for the significantly regulated genes.
    
    :param log2fold_dataframe
        :type pd.DataFrame
    :param pval_dataframe
        :type pd.DataFrame
    :param annotation_dataframe --> needs the column KO_definition
        :type pd.DataFrame
    :param savep
        :type str
    :param title
        :type str
    
    :returns 0
        :type int

'''
def plot_goHeatmap_html(log2fold_dataframe:pd.DataFrame,
                        pval_dataframe:pd.DataFrame,
                        annotation_dataframe:pd.DataFrame,
                        savep:str,title="Significant Log2Fold values across experiments")->int:
    try:
        annot_mat = np.repeat(
            annotation_dataframe["PFAM_NAME"].to_numpy()[np.newaxis, :], 
            len(list(log2fold_dataframe.iloc[:,1:].columns)), axis=0
        ).T
        
        custom_yticklabels = pval_dataframe["gene_id"]
        custom_xticklabels = list(log2fold_dataframe.iloc[:,1:].columns)
        
        matrix = log2fold_dataframe.iloc[:,1:].to_numpy()
        matrix = np.nan_to_num(matrix, nan=0)

        pmat = pval_dataframe.iloc[:,1:].to_numpy()
        pmat = np.nan_to_num(pmat, nan=1)
        
        # mask everything that is not significant
        mask_matrix = pmat >= 0.05 
        masked_matrix = np.where(mask_matrix, np.nan, matrix)

        hover_text = list(annot_mat)
        heatmap_masked = go.Heatmap(z=masked_matrix,
                                    hovertext=hover_text,
                                    hoverinfo='text',
                                    colorscale='viridis', 
                                    x=np.arange(len(matrix[0])),
                                    y=custom_yticklabels)
        heatmap_original = go.Heatmap(z=matrix,
                                      visible='legendonly', 
                                      colorscale='viridis',
                                      x=np.arange(len(matrix[0])),
                                      y=custom_yticklabels)
        
        # Create the figure
        fig = go.Figure(data=[heatmap_masked, heatmap_original])

        # Update layout to set axis titles and hover mode
        fig.update_layout(
            title=title,
            xaxis=dict(
                title='Log2FoldChange',
                tickmode='array', 
                tickvals=np.arange(len(custom_xticklabels)), 
                ticktext=custom_xticklabels, 
                showgrid=False, 
                tickangle=45),
            
            yaxis=dict(
                title='Hydra Gene Identifier', 
                tickmode='array',
                tickvals=np.arange(len(custom_yticklabels)), 
                ticktext=custom_yticklabels,
                showgrid=False),
            
            hovermode='closest',
        )
        
        # save the plot
        write_html(fig, savep)
        return 0
    except Exception as e:
        raise Exception("[-] ERROR producing goHeatmap with exception: {}".format(e))

In [30]:
# iterate over all celltypes and produce heatmaps
for celltype in celltype_dict.keys():
    result_path_png_heatmap = "../results/figures/genes_log2Fold_heatmaps/" + celltype + "_log2fold.png"
    result_path_html_heatmap = "../results/figures/genes_log2Fold_heatmaps/" + celltype + "_heatmap.html"
    genes_to_exp, genes_to_pval, annotation_genes = extract_target_gene_expressions_and_annotations(diff_exp_table,
                                                                                                      p_value_table, 
                                                                                                      annotation_table, 
                                                                                                      celltype_dict[celltype])
    plot_seaborn_heatmap(genes_to_exp, genes_to_pval, result_path_png_heatmap)
    plot_goHeatmap_html(genes_to_exp, genes_to_pval, annotation_genes, result_path_html_heatmap)

[*] Extracting 29 target genes ...
[*] Length after extracting target genes: 29 ...
[*] Extracting KO definition line for annotations ...
[+] DONE
[*] Starting heatmap plotting procedure ...
[*] Saving heatmap into: ../results/figures/genes_log2Fold_heatmaps/Ec_Head_log2fold.png
[+] DONE
[*] Extracting 11 target genes ...
[*] Length after extracting target genes: 11 ...
[*] Extracting KO definition line for annotations ...
[+] DONE
[*] Starting heatmap plotting procedure ...
[*] Saving heatmap into: ../results/figures/genes_log2Fold_heatmaps/En_Head_log2fold.png
[+] DONE
[*] Extracting 3 target genes ...
[*] Length after extracting target genes: 3 ...
[*] Extracting KO definition line for annotations ...
[+] DONE
[*] Starting heatmap plotting procedure ...
[*] Saving heatmap into: ../results/figures/genes_log2Fold_heatmaps/Ec_BodyCol.SC_log2fold.png
[+] DONE
[*] Extracting 1 target genes ...
[*] Length after extracting target genes: 1 ...
[*] Extracting KO definition line for annotatio

[+] DONE
[*] Extracting 16 target genes ...
[*] Length after extracting target genes: 16 ...
[*] Extracting KO definition line for annotations ...
[+] DONE
[*] Starting heatmap plotting procedure ...
[*] Saving heatmap into: ../results/figures/genes_log2Fold_heatmaps/I_Ec1.5N_log2fold.png
[+] DONE
[*] Extracting 13 target genes ...
[*] Length after extracting target genes: 13 ...
[*] Extracting KO definition line for annotations ...
[+] DONE
[*] Starting heatmap plotting procedure ...
[*] Saving heatmap into: ../results/figures/genes_log2Fold_heatmaps/I_En1N_log2fold.png
[+] DONE
[*] Extracting 12 target genes ...
[*] Length after extracting target genes: 12 ...
[*] Extracting KO definition line for annotations ...
[+] DONE
[*] Starting heatmap plotting procedure ...
[*] Saving heatmap into: ../results/figures/genes_log2Fold_heatmaps/I_En3N_log2fold.png
[+] DONE
[*] Extracting 437 target genes ...
[*] Length after extracting target genes: 437 ...
[*] Extracting KO definition line for a

In [31]:
plot_goHeatmap_html(wnt_genes_to_exp, wnt_genes_to_pval, 
                    annotation_genes_to_wnt, "../results/figures/genes_log2Fold_heatmaps/wnt_heatmap.html")

plot_goHeatmap_html(frizzled_genes_to_exp, frizzled_genes_to_pval, 
                    annotation_genes_to_frizzled, "../results/figures/genes_log2Fold_heatmaps/frizzled_heatmap.html")

plot_goHeatmap_html(catenin_genes_to_exp, catenin_genes_to_pval, 
                    annotation_genes_to_catenin, "../results/figures/genes_log2Fold_heatmaps/catenin_heatmap.html")

plot_goHeatmap_html(lectin_genes_to_exp, lectin_genes_to_pval, 
                    annotaion_genes_to_lectin, "../results/figures/genes_log2Fold_heatmaps/lectins_heatmap.html")

0